# Systems to Blend Different Feed Streams

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import control as con
import pandas as pd

from bounded_random_walk import sample_bounded_random_walk

from cas_models.transformations import connect_systems
from cas_models.continuous_time.models import StateSpaceModelCT
from feed_conc_ctrl.models import MixingTankModelCT, FlowMixerCT, RatioControlledFlowMixerCT
from cas_models.discrete_time.models import StateSpaceModelDTFromCTRK4
from cas_models.discrete_time.simulate import make_n_step_simulation_function_from_model
from feed_conc_ctrl.plot_utils import make_tsplots

In [ ]:
plot_dir = Path("./plots")
plot_dir.mkdir(exist_ok=True)

FIGSIZE = (5.5, 5)

## Generate Bounded Random Walk (BRW) Sequences

In [ ]:
seed = 100
rng = np.random.default_rng(seed)

# Noise std. dev.
sd_e = 5.0

# Bounded random walk parameters
r1 = -40.0  # When x = r1, bias = +1 (pushes up)
r2 = 40.0  # When x = r2, bias = -1 (pushes down)
a1 = 0.2  # aggressiveness of lower bound
a2 = 0.2  # aggressiveness of upper bound

# Number of random walks to generate
n_walks = 3

nT = 600
bounded_random_walks = sample_bounded_random_walk(sd_e, r1, r2, a1, a2, nT, n_walks=3, rng=rng)
assert bounded_random_walks.shape == (nT, n_walks)

# Nominal input value
u_nop = 50.0

# Time vector
Ts = 1.0
t = Ts * np.arange(nT)

In [ ]:
# Only plot the first t_stop minutes of each BRW
t_stop = 600.0
nT_plot = int(np.floor(t_stop / Ts))

n_plots = min(5, n_walks)

marker = ""

fig, axes = plt.subplots(n_plots, 1, sharex=True, figsize=(7, 1 + 1.5*n_plots))

for i, ax in enumerate(axes):
    u = bounded_random_walks[:, i]
    ax.plot(t[:nT_plot], u_nop + u[:nT_plot], marker=marker)
    ax.axhline(u_nop + r1, linestyle='--', color='grey', label='min')
    ax.axhline(u_nop + r2, linestyle='--', color='grey', label='max')
    ax.set_ylim([u_nop + 1.3 * r1, u_nop + 1.3 * r2])
    ax.set_ylabel("%")
    ax.grid()
    ax.set_title(f"Bounded Random Walk {i+1:d}")

ax.set_xlabel("Time (mins)")
plt.tight_layout()
filename = "bounded_random_walks.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()

In [ ]:
# Convert two BRWs into independent concentration disturbances
c_bounds =  [(20, 40), (60, 80)]
c_nom = [np.mean(c_bounds[0]), np.mean(c_bounds[1])]  # Nominal concentrations of input streams 1 and 2

c_1 = c_nom[0] + bounded_random_walks[:, 0] * (np.diff(c_bounds[0])) / 50.0
c_2 = c_nom[1] + bounded_random_walks[:, 1] * (np.diff(c_bounds[1])) / 50.0

assert c_1.shape == (nT,)
assert c_2.shape == (nT,)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.5))

ax.plot(t[:nT_plot], c_1[:nT_plot], marker=marker, label='c_1')
ax.plot(t[:nT_plot], c_2[:nT_plot], marker=marker, label='c_2')
for i in range(2):
    ax.fill_between(t[:nT_plot], c_bounds[i][0], c_bounds[i][1], color=f"C{i}", alpha=0.1, label=f"c_{i+1} bounds")
ax.set_ylim([0, 100])
ax.set_xlabel("Time (mins)")
ax.set_ylabel("Composition (%)")
ax.grid()
ax.set_title(f"Bounded Random Walks")

plt.tight_layout()
filename = "mixing_bounded_random_walks.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()

## Construct Mixer and Tank Dynamic System

In [ ]:
def print_sys_dimensions(sys):
    print(sys.name, f"({sys.ny}x{sys.nu})")
    for attr_name in ["input_names", "state_names", "output_names"]:
        print(f"{attr_name:>15s}: {getattr(sys, attr_name)}")

In [ ]:
# Tank dimensions
H = 9   # Height [m]
A = 10  # Cross-sectional area [m^2]

# Diameter of the tank [m]
D = np.sqrt(4 * A / np.pi)

tank_model = MixingTankModelCT(D=D, name="tank")
print_sys_dimensions(tank_model)

In [ ]:
# mixer_model = RatioControlledFlowMixerCT(2, name="mixer")
# print_sys_dimensions(mixer_model)

# Simple 2-input flow mixer
mixer_model = FlowMixerCT(2, name="mixer")
print_sys_dimensions(mixer_model)

In [ ]:
# connections = {
#     "mixer_v_dot_out": "tank_v_dot_in",
#     'tank_conc_in': 'mixer_conc_out',
# }
connections = {
    'tank_v_dot_in': 'mixer_v_dot_out' ,
    'tank_conc_in': 'mixer_conc_out',
}

model_class = StateSpaceModelCT
mixer_tank_system = connect_systems(
    [mixer_model, tank_model],
    connections,
    model_class,
    name="mixer_tank_system",
    verbose_names=True,
)
print_sys_dimensions(mixer_tank_system)

In [ ]:
dt = Ts
mixer_tank_system_dt = StateSpaceModelDTFromCTRK4(mixer_tank_system, dt)
print_sys_dimensions(mixer_tank_system_dt)

In [ ]:
simulate = make_n_step_simulation_function_from_model(mixer_tank_system_dt, nT)
simulate

In [ ]:
# Input sequence
mixer_v_dot_in_1 = np.full(nT, 0.5)
mixer_conc_in_1 = c_1  # np.full(nT, c_nom[0])
mixer_v_dot_in_2 = np.full(nT, 0.5)
mixer_conc_in_2 = c_2  # np.full(nT, c_nom[1])
tank_v_dot_out = mixer_v_dot_in_1 + mixer_v_dot_in_2

U = np.stack([
    mixer_v_dot_in_1,
    mixer_conc_in_1,
    mixer_v_dot_in_2,
    mixer_conc_in_2,
    tank_v_dot_out
]).T

assert U.shape == (nT, mixer_tank_system_dt.nu)

In [ ]:
# Initial condition
tank_L = H  # Start full
tank_m = 4494.27
x0 = [tank_L, tank_m]

# Simulation output time vector
t_eval = Ts * np.arange(nT + 1)

assert t_eval.shape == (nT + 1, )
X, Y = simulate(t_eval, U, x0)
assert X.shape == (nT + 1, mixer_tank_system_dt.n)
assert Y.shape == (nT + 1, mixer_tank_system_dt.ny)

sim_results = pd.concat(
    [
        pd.DataFrame(t_eval, columns=["time"]),
        pd.DataFrame(U, columns=mixer_tank_system_dt.input_names),
        pd.DataFrame(Y, columns=mixer_tank_system_dt.output_names),
    ],
    axis=1,
)
sim_results

In [ ]:
flow_units = "m^3/min"
conc_units = "%"
level_units = "m"
mass_units = "kg"

units = {
    "mixer_v_dot_in_1": flow_units,
    "mixer_conc_in_1": conc_units,
    "mixer_v_dot_in_2": flow_units,
    "mixer_conc_in_2": conc_units,
    "tank_v_dot_out": flow_units,
    "mixer_v_dot_out": flow_units,
    "mixer_conc_out": conc_units,
    "tank_L": level_units,
    "tank_m": mass_units,
    "tank_conc_out": conc_units
}

# Define plot structure
plot_info = {
    # "Tank Level": {
    #     "Tank": {"var_name": "tank_L"},
    # },
    "Concentrations": {
        "Feed 1": {"var_name": "mixer_conc_in_1"},
        "Feed 2": {"var_name": "mixer_conc_in_2"},
        "Mixer outlet": {"var_name": "mixer_conc_out"},
        "Tank outlet": {"var_name": "tank_conc_out"},
    },
    "Flow Rates": {
        "Feed 1": {"var_name": "mixer_v_dot_in_1"},
        "Feed 2": {"var_name": "mixer_v_dot_in_2"},
        "Mixer Out": {"var_name": "mixer_v_dot_out"},
        "Tank outlet": {"var_name": "tank_v_dot_out"},
    },
}

# Create plots
fig, axes = make_tsplots(sim_results, plot_info, units=units)

plt.tight_layout()
filename = "blending_sim_no_ctrl.png"
plt.savefig(plot_dir / filename)